In [1]:
import torch
import torch.nn.functional as F

# Batch size = 1, Sequence length = 4, Embedding dimension = 2
X = torch.tensor([[
    [1.0, 0.0],  # x1
    [0.0, 1.0],  # x2
    [1.0, 1.0],  # x3
    [2.0, 0.0],  # x4
]])  # Shape: (1, 4, 2)

# Step 1: Compute raw dot products: w'_ij = x_i^T x_j
# X shape: (b, t, k), need to transpose last two dims to get (b, k, t)
X_transpose = X.transpose(1, 2)  # Shape: (1, 2, 4)
dot_products = torch.bmm(X, X_transpose)  # Shape: (1, 4, 4)

# Step 2: Softmax over last dim (j) to get attention weights
attention_weights = F.softmax(dot_products, dim=-1)  # Shape: (1, 4, 4)

# Step 3: Weighted sum over the input sequence
# attention_weights @ X = (b, t, t) @ (b, t, k) => (b, t, k)
Y = torch.bmm(attention_weights, X)  # Output: (1, 4, 2)

# View results
print("Input X (b=1, t=4, k=2):\n", X)
print("\nDot Products (X @ Xᵀ):\n", dot_products)
print("\nAttention Weights (Softmax):\n", attention_weights)
print("\nOutput Y (Self-attended vectors):\n", Y)

Input X (b=1, t=4, k=2):
 tensor([[[1., 0.],
         [0., 1.],
         [1., 1.],
         [2., 0.]]])

Dot Products (X @ Xᵀ):
 tensor([[[1., 0., 1., 2.],
         [0., 1., 1., 0.],
         [1., 1., 2., 2.],
         [2., 0., 2., 4.]]])

Attention Weights (Softmax):
 tensor([[[0.1966, 0.0723, 0.1966, 0.5344],
         [0.1345, 0.3655, 0.3655, 0.1345],
         [0.1345, 0.1345, 0.3655, 0.3655],
         [0.1050, 0.0142, 0.1050, 0.7758]]])

Output Y (Self-attended vectors):
 tensor([[[1.4621, 0.2689],
         [0.7689, 0.7311],
         [1.2311, 0.5000],
         [1.7616, 0.1192]]])


In [2]:
# Hyperparameters
b, t, k = 1, 4, 2  # batch, time (tokens), embedding size
dk = k ** 0.5      # scaling factor

# Step 1: Input (batch of 4 tokens, each 2D)
X = torch.tensor([[
    [1.0, 0.0],  # x1
    [0.0, 1.0],  # x2
    [1.0, 1.0],  # x3
    [2.0, 0.0],  # x4
]], requires_grad=False)  # shape: (1, 4, 2)

# Step 2: Learnable weight matrices (simulate W_q, W_k, W_v)
W_q = torch.tensor([[1.0, 0.5], [0.5, 1.0]])  # (2, 2)
W_k = torch.tensor([[0.5, 1.0], [1.0, 0.5]])  # (2, 2)
W_v = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # (2, 2)

# Step 3: Compute Q, K, V
Q = X @ W_q     # shape: (1, 4, 2)
K = X @ W_k     # shape: (1, 4, 2)
V = X @ W_v     # shape: (1, 4, 2) = same as X here

# Step 4: Attention scores: Q @ K^T
K_T = K.transpose(1, 2)  # (1, 2, 4)
scores = (Q @ K_T) / dk  # shape: (1, 4, 4)

# Step 5: Softmax over j
weights = F.softmax(scores, dim=-1)  # shape: (1, 4, 4)

# Step 6: Weighted sum: attention output
Y = weights @ V  # shape: (1, 4, 2)

# Print everything
print("Input X:\n", X)
print("\nQuery Q:\n", Q)
print("\nKey K:\n", K)
print("\nValue V:\n", V)
print("\nAttention Scores (QKᵀ / sqrt(dk)):\n", scores)
print("\nAttention Weights (Softmax):\n", weights)
print("\nOutput Y (Self-attended):\n", Y)

Input X:
 tensor([[[1., 0.],
         [0., 1.],
         [1., 1.],
         [2., 0.]]])

Query Q:
 tensor([[[1.0000, 0.5000],
         [0.5000, 1.0000],
         [1.5000, 1.5000],
         [2.0000, 1.0000]]])

Key K:
 tensor([[[0.5000, 1.0000],
         [1.0000, 0.5000],
         [1.5000, 1.5000],
         [1.0000, 2.0000]]])

Value V:
 tensor([[[1., 0.],
         [0., 1.],
         [1., 1.],
         [2., 0.]]])

Attention Scores (QKᵀ / sqrt(dk)):
 tensor([[[0.7071, 0.8839, 1.5910, 1.4142],
         [0.8839, 0.7071, 1.5910, 1.7678],
         [1.5910, 1.5910, 3.1820, 3.1820],
         [1.4142, 1.7678, 3.1820, 2.8284]]])

Attention Weights (Softmax):
 tensor([[[0.1506, 0.1797, 0.3644, 0.3054],
         [0.1591, 0.1333, 0.3226, 0.3850],
         [0.0846, 0.0846, 0.4154, 0.4154],
         [0.0807, 0.1149, 0.4726, 0.3318]]])

Output Y (Self-attended):
 tensor([[[1.1257, 0.5441],
         [1.2517, 0.4559],
         [1.3308, 0.5000],
         [1.2170, 0.5875]]])
